In [18]:
#Read the spell file
import pandas
import re

spell_df = pandas.read_csv("spells.csv", sep=";")

# This file is full of spells, but some of them have 'Unknown' or 'empty' Incantations. For the sake of simplicity, we want to use only the spells with known, verbal incantations. so we get rid of the rest.
spell_df = spell_df[(spell_df["Incantation"] != "Unknown") & (spell_df["Incantation"] != "empty")]

# Normalize spell incantations to lowercase for case-insensitive comparison
spell_df["Incantation"] = spell_df["Incantation"].str.lower()

# Now we have a list of spells with known incantations. We also have the full script for the first 3 HP movies. Now, we want to find how often a spell is said in general, but also for each given character.
# Import the full script file.
script_df = pandas.read_csv("Scripts Merged.csv", sep=";")

# Normalize character names to lowercase for case-insensitive comparison
script_df["Character"] = script_df["Character"].str.lower()

# We want to sieve through the script, for mentions of the incantations. Keep track of which spell is being used, and by who. also keep track of the unique spells total usage.
# Create a dictionary to keep track of the spells and their usage.
spell_usage = {}
character_spell_usage = {}

for spell in spell_df["Incantation"].dropna():
    # Use word boundaries to match the spell as a complete word, optionally followed by punctuation (!, ?, .)
    # This ensures "pack" matches "pack!", "pack?", "pack." but not "packing" or "packed"
    spell_pattern = r'\b' + re.escape(spell) + r'\b'
    total_usage = script_df["Sentence"].str.contains(spell_pattern, case=False, na=False, regex=True).sum()
    spell_usage[spell] = int(total_usage)

    for character in script_df["Character"].dropna().unique():
        character_usage = script_df.loc[
            script_df["Character"].eq(character),
            "Sentence"
        ].str.contains(spell_pattern, case=False, na=False, regex=True).sum()

        if character not in character_spell_usage:
            character_spell_usage[character] = {}

        character_spell_usage[character][spell] = int(character_usage)
        

In [21]:
# Display most used spells, and most cast-happy wizards

# Sort spells by usage
sorted_spells = sorted(spell_usage.items(), key=lambda x: x[1], reverse=True)
print("Most used spells:")
for spell, usage in sorted_spells[:10]:  # Display top 10 spells
    print(f"{spell}: {usage} times")
# Sort characters by total spell usage
character_totals = {char: sum(spells.values()) for char, spells in character_spell_usage.items()}
sorted_characters = sorted(character_totals.items(), key=lambda x: x[1], reverse=True)
print("\nMost spell-casting characters:")
for character, total in sorted_characters[:10]:  # Display top 10 characters
    print(f"{character}: {total} spells cast")

Most used spells:
riddikulus: 8 times
expecto patronum: 7 times
expelliarmus: 5 times
lumos: 5 times
alohomora: 4 times
wingardium leviosa: 4 times
arania exumai: 3 times
lumos maxima: 3 times
vera verto: 3 times
immobulus: 2 times

Most spell-casting characters:
harry: 22 spells cast
hermione: 9 spells cast
lupin: 6 spells cast
ron: 4 spells cast
snape: 3 spells cast
gilderoy lockhart: 3 spells cast
mcgonagall: 2 spells cast
class: 2 spells cast
draco: 2 spells cast
tom riddle: 2 spells cast
